# 03.01 · Autograd from scratch (micrograd)

**Chapter:** Deep Learning · **Milestone:** M1

## Goals
- [ ] Build a scalar autograd engine: every operation remembers how to pass gradients back
- [ ] Understand backprop as the chain rule applied in reverse topological order
- [ ] Train a tiny neural net with it and verify gradients against PyTorch

## Resources
- [Karpathy – The spelled-out intro to neural networks and backpropagation](https://www.youtube.com/watch?v=VMj-3S1tku0) — free (watch this alongside the notebook)
- [karpathy/micrograd](https://github.com/karpathy/micrograd) — free

PyTorch's `loss.backward()` is this lesson, vectorised and written in C++/CUDA. After this you'll never treat it as magic.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np

## 1. The `Value` class
Each `Value` stores data, a gradient, its parents, and a `_backward` closure that applies the **local** chain-rule step:
"given my output's gradient, add my contribution to my inputs' gradients".

In [ ]:
class Value:
    def __init__(self, data, _parents=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._parents = set(_parents)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad  # d(a+b)/da = 1
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad  # d(a*b)/da = b
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):  # k is a plain number
        out = Value(self.data**k, (self,), f"**{k}")
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # topological order: every node after all of its inputs
        order, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    build(p)
                order.append(v)
        build(self)
        self.grad = 1.0  # d(self)/d(self)
        for v in reversed(order):
            v._backward()

    # conveniences so expressions read naturally
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __repr__(self): return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

## 2. Check against PyTorch
Note `+=` in every `_backward`: if a value is used twice (like `a` below), its gradients from both uses must **add up**.

In [ ]:
import torch

a, b = Value(2.0), Value(-3.0)
c = a * b + a**2
d = (c / 4).tanh() + a
d.backward()

ta, tb = torch.tensor(2.0, requires_grad=True), torch.tensor(-3.0, requires_grad=True)
td = torch.tanh((ta * tb + ta**2) / 4) + ta
td.backward()
print("ours :", a.grad, b.grad)
print("torch:", ta.grad.item(), tb.grad.item())
assert abs(a.grad - ta.grad.item()) < 1e-6 and abs(b.grad - tb.grad.item()) < 1e-6

## 3. Neurons, layers, MLP
A neuron: $\tanh(\sum_i w_i x_i + b)$. A layer: many neurons. An MLP: layers stacked.

In [ ]:
class Neuron:
    def __init__(self, n_in):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)
    def __call__(self, x, act=True):
        out = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return out.tanh() if act else out
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, n_in, n_out):
        self.neurons = [Neuron(n_in) for _ in range(n_out)]
    def __call__(self, x, act=True):
        return [n(x, act) for n in self.neurons]
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, sizes):
        self.layers = [Layer(a, b) for a, b in zip(sizes, sizes[1:])]
    def __call__(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x, act=i < len(self.layers) - 1)  # last layer: raw score
        return x[0]
    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]

## 4. Train on "two moons"
Labels are ±1; loss is the hinge loss `max(0, 1 - y·score)` averaged, plus a little L2 regularisation.

In [ ]:
from sklearn.datasets import make_moons

random.seed(0)
X, y = make_moons(n_samples=60, noise=0.1, random_state=0)
y = y * 2 - 1  # {0,1} -> {-1,+1}
model = MLP([2, 16, 16, 1])
print("parameters:", len(model.parameters()))

def relu(v):  # max(0, v) built from our ops
    return v * (1.0 if v.data > 0 else 0.0)

history = []
for step in range(40):
    scores = [model(list(map(Value, xi))) for xi in X]
    data_loss = sum(relu(1 - float(yi) * s) for yi, s in zip(y, scores)) / len(y)
    reg = 1e-4 * sum(p * p for p in model.parameters())
    loss = data_loss + reg
    acc = np.mean([(s.data > 0) == (yi > 0) for yi, s in zip(y, scores)])

    for p in model.parameters():
        p.grad = 0.0  # gradients accumulate: zero them every step (same in PyTorch!)
    loss.backward()
    lr = 0.5 - 0.45 * step / 40
    for p in model.parameters():
        p.data -= lr * p.grad
    history.append((loss.data, acc))

print(f"final loss {history[-1][0]:.3f}, accuracy {history[-1][1]:.0%}")

In [ ]:
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 50), np.linspace(-1, 1.5, 50))
Z = np.array([model([Value(a), Value(b)]).data > 0 for a, b in zip(xx.ravel(), yy.ravel())]).reshape(xx.shape)
plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm"); plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k")
plt.title("decision boundary learned by our 100%-Python autograd"); plt.show()

## Exercises
1. Close this notebook and re-implement `Value` (+, *, tanh, backward) from memory. Then compare.
2. Add `exp()` and `log()` to `Value`, implement softmax + cross-entropy, and check gradients against PyTorch.
3. Time one training step. Count how many `Value` objects one step creates. Why is real autograd vectorised over tensors?

In [ ]:
# your code here